# Etap 2 — wartości zdarzeń

Masa każdego odsypu liczona 9 wariantami (najbliższa próbka, mediana, średnia z okna 3 s × przerwa po końcu sygnału d = 0, 2, 3 s) i masy zasypów dwoma sposobami; błąd metod liczy etap 3 (`notebooks/errors/`). Ustalenia: `praca-inzynierska/zakres.md` §7 „Etap 2 — ustalenia”, plan `plans/dms-data-analysis/etap2-wartosci-zdarzen-plan.md`.

Dane: `data/results/<obiekt>/<id>/event_values.db` z `uv run python -m dms_analysis.events compute` (po `prepare`). **Wariant podstawowy:** mediana 3 s, d = 3 s.

Wyniki niezwalidowane, do przeglądu.

In [ ]:
from IPython.display import Markdown, display

from dms_analysis import style
from dms_analysis.data.catalog import load_catalog
from dms_analysis.events import browser, views

style.register()
results = views.load_all(load_catalog())
for warning in views.warnings(results):
    display(Markdown(f"<span style='color:#c0392b'><b>Uwaga:</b> {warning}</span>"))

## 1. Odsypy w okresach

Z analizy wypadają tylko odsypy, których okno (3 s przed, do 6 s po końcu) nachodzi na inne zdarzenie, lukę, granicę części okresu albo problem z katalogu (**okno**), oraz odsypy nachodzące na zasyp (**kolizja**, analizowane osobno później). Odsypy bez spadku wskazań zostają w analizie.

In [ ]:
views.summary_table(results)

## 2. Przeglądarka odsypu

Wybór okresu i doby, kolejność: po kolei w czasie, największa rozbieżność między wariantami, tylko odsypy bez spadku wskazań (w którymkolwiek wariancie), największa masa. Po prawej masy 9 wariantów: ramka = wariant podstawowy, kolor = różnica względem niego.

In [ ]:
browser.show(results)

## 3. Zestawienie mas odsypów

Masy w wariancie podstawowym, osobno dla każdego okresu, potem różnice pozostałych wariantów względem podstawowego (jak często i o ile wariant zmienia wynik).

In [ ]:
views.mass_histograms(results).show()

In [ ]:
views.variant_differences(results).show()

In [ ]:
views.no_drop_table(results)

## 4. Zasypy

Wszystkie zasypy są wpisane ręcznie (granice z marginesem). Masa = wskazanie po − wskazanie przed: z pojedynczych próbek i z mediany 3 s. Odsypy w trakcie zasypu zmniejszają masę liczoną z różnicy wskazań (ważne dla bilansu w etapie 3). Na wykresach: niebieskie pasmo = zasyp, szare pasma = odsypy.

In [ ]:
views.refills_table(results)

In [ ]:
for full_id, item in results.items():
    for figure in views.refill_figures(full_id, item):
        figure.show()

## 5. Granice dnia i nocy

Godziny pierwszego i ostatniego odsypu każdej doby — podstawa granic dnia (5–22) używanych w etapie 3 (`dms_analysis.errors.fragments`: `DAY_START_H`, `DAY_END_H`). Okres `swietochlowice-1/0002-2026-09-28` ma czas szacowany (±2 h), więc pokazany osobnym kolorem. Test półsyntetyczny z rampą usunięty 2026-11-20: zastępuje go test na siatce w `notebooks/errors/single_dispense.ipynb`.

In [ ]:
views.first_last_hours(results).show()